In [1]:
import mygene
import pandas as pd
import json

# Load our 1000 gene list
gene_list = json.load(open(
    '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival/models/gene_list.json'))

# Convert gene symbols to Ensembl IDs
mg = mygene.MyGeneInfo()
print(f"Converting {len(gene_list)} gene symbols to Ensembl IDs...")

result = mg.querymany(gene_list,
                       scopes='symbol',
                       fields='ensembl.gene',
                       species='human',
                       returnall=True)

symbol_to_ensembl = {}
for r in result['out']:
    if 'ensembl' in r:
        ensembl = r['ensembl']
        # ensembl field can be a list or dict
        if isinstance(ensembl, list):
            symbol_to_ensembl[r['query']] = ensembl[0]['gene']
        elif isinstance(ensembl, dict):
            symbol_to_ensembl[r['query']] = ensembl['gene']

print(f"Successfully mapped: {len(symbol_to_ensembl)} / {len(gene_list)} genes")
print(f"EGFR -> {symbol_to_ensembl.get('EGFR', 'not found')}")
print(f"TP53 -> {symbol_to_ensembl.get('TP53', 'not found')}")
print(f"LINGO2 -> {symbol_to_ensembl.get('LINGO2', 'not found')}")

Converting 1000 gene symbols to Ensembl IDs...


16 input query terms found dup hits:	[('RRN3P1', 2), ('TRPC2', 2), ('NAPSB', 2), ('MST1P2', 2), ('ABCC6P1', 2), ('ABCA17P', 2), ('FAM95B1
169 input query terms found no hit:	['LOC441869', 'BZRAP1', 'GPR172B', 'C20orf197', 'CMAH', 'GYLTL1B', 'C12orf39', 'LOC283070', 'ARNTL2'


Successfully mapped: 828 / 1000 genes
EGFR -> not found
TP53 -> not found
LINGO2 -> ENSG00000174482


In [4]:
# Read pan-cancer gene index only (no data yet)
print("Reading pan-cancer gene index...")
pancancer_index = pd.read_csv(
    '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival/data/external/pancancer_tpm',
    sep='\t', index_col=0, nrows=0).columns

# Actually read the row index
pancancer_genes_raw = pd.read_csv(
    '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival/data/external/pancancer_tpm',
    sep='\t', usecols=[0], header=0).iloc[:, 0]

# Strip version numbers (ENSG00000146648.15 -> ENSG00000146648)
pancancer_genes_clean = set(g.split('.')[0] for g in pancancer_genes_raw)

print(f"Pan-cancer genes (after stripping versions): {len(pancancer_genes_clean)}")

# Check overlap
our_ensembl = set(symbol_to_ensembl.values())
overlap = our_ensembl.intersection(pancancer_genes_clean)

print(f"Our mapped Ensembl IDs: {len(our_ensembl)}")
print(f"Overlap with pan-cancer: {len(overlap)}")
print(f"\nExample overlapping genes:")
for sym, ens in list(symbol_to_ensembl.items())[:5]:
    status = "✅" if ens in pancancer_genes_clean else "❌"
    print(f"  {sym} -> {ens} {status}")

Reading pan-cancer gene index...
Pan-cancer genes (after stripping versions): 60498
Our mapped Ensembl IDs: 828
Overlap with pan-cancer: 790

Example overlapping genes:
  LINGO2 -> ENSG00000174482 ✅
  EPGN -> ENSG00000182585 ✅
  DKK1 -> ENSG00000107984 ✅
  CD109 -> ENSG00000156535 ✅
  IRX5 -> ENSG00000176842 ✅


In [5]:
base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'

# Build reverse mapping: ensembl_with_version -> gene_symbol
print("Building ID mapping...")

pancancer_genes_versioned = pd.read_csv(
    f'{base}/data/external/pancancer_tpm',
    sep='\t', usecols=[0], header=0).iloc[:, 0].tolist()

ensembl_to_symbol = {v: k for k, v in symbol_to_ensembl.items()}
versioned_to_symbol = {}
for g in pancancer_genes_versioned:
    clean = g.split('.')[0]
    if clean in ensembl_to_symbol:
        versioned_to_symbol[g] = ensembl_to_symbol[clean]

print(f"Genes to extract: {len(versioned_to_symbol)}")
print(f"Example mappings:")
for k, v in list(versioned_to_symbol.items())[:5]:
    print(f"  {k} -> {v}")

with open(f'{base}/models/ensembl_to_symbol.json', 'w') as f:
    json.dump(versioned_to_symbol, f, indent=2)

print("\nSaved: models/ensembl_to_symbol.json")

Building ID mapping...
Genes to extract: 790
Example mappings:
  ENSG00000158486.13 -> DNAH3
  ENSG00000172137.18 -> CALB2
  ENSG00000206072.12 -> SERPINB11
  ENSG00000197421.9 -> GGT3P
  ENSG00000057294.13 -> PKP2

Saved: models/ensembl_to_symbol.json


In [6]:
base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'

print("Reading pan-cancer expression for our 790 genes...")
print("This will take 3-5 minutes — large file...")

# Read full file but only keep our rows
pancancer_raw = pd.read_csv(
    f'{base}/data/external/pancancer_tpm',
    sep='\t', index_col=0)

print(f"Full pan-cancer shape: {pancancer_raw.shape}")

# Filter to our 790 genes
our_versioned_ids = list(versioned_to_symbol.keys())
pancancer_filtered = pancancer_raw.loc[
    pancancer_raw.index.isin(our_versioned_ids)
]

print(f"Filtered shape: {pancancer_filtered.shape}")

# Rename index from Ensembl to gene symbols
pancancer_filtered.index = [versioned_to_symbol[g] for g in pancancer_filtered.index]
pancancer_filtered.index.name = 'gene'

# Transpose so rows=patients, columns=genes
pancancer_filtered = pancancer_filtered.T
print(f"After transpose (patients × genes): {pancancer_filtered.shape}")

# Save
pancancer_filtered.to_csv(f'{base}/data/external/pancancer_790genes.csv')
print(f"\nSaved: data/external/pancancer_790genes.csv")
print(f"Shape: {pancancer_filtered.shape}")
print(f"Value range: {pancancer_filtered.values.min():.2f} to {pancancer_filtered.values.max():.2f}")

Reading pan-cancer expression for our 790 genes...
This will take 3-5 minutes — large file...
Full pan-cancer shape: (60498, 10535)
Filtered shape: (790, 10535)
After transpose (patients × genes): (10535, 790)

Saved: data/external/pancancer_790genes.csv
Shape: (10535, 790)
Value range: -9.97 to 17.11


In [7]:
import numpy as np

base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'

pancancer = pd.read_csv(f'{base}/data/external/pancancer_790genes.csv', index_col=0)

print(f"Shape: {pancancer.shape}")
print(f"Any NaN: {pancancer.isna().any().any()}")
print(f"Any inf: {np.isinf(pancancer.values).any()}")
print(f"\nValue distribution:")
print(f"  Min:    {pancancer.values.min():.3f}")
print(f"  Max:    {pancancer.values.max():.3f}")
print(f"  Mean:   {pancancer.values.mean():.3f}")
print(f"  Median: {np.median(pancancer.values):.3f}")

# Check overlap with our LUAD expression matrix
expr = pd.read_csv(f'{base}/data/processed/expression_matrix.csv', index_col=0)
common_genes = set(pancancer.columns).intersection(set(expr.columns))
print(f"\nGenes in pan-cancer: {len(pancancer.columns)}")
print(f"Genes in our LUAD:   {len(expr.columns)}")
print(f"Common genes:        {len(common_genes)}")

# Check if any LUAD patients leaked into pan-cancer
luad_ids = set(expr.index)
pancancer_ids = set(pancancer.index)
overlap_patients = luad_ids.intersection(pancancer_ids)
print(f"\nLUAD patients in pan-cancer: {len(overlap_patients)}")
print(f"(These must be removed before pretraining)")

Shape: (10535, 790)
Any NaN: False
Any inf: False

Value distribution:
  Min:    -9.966
  Max:    17.108
  Mean:   -0.881
  Median: -0.512

Genes in pan-cancer: 790
Genes in our LUAD:   1000
Common genes:        790

LUAD patients in pan-cancer: 0
(These must be removed before pretraining)


In [8]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

class ExpressionAutoencoder(nn.Module):
    """
    Autoencoder for pretraining the expression encoder.
    
    Encoder: 790 → 256 → 64 → 32  (same as FusionModelV3 encoder)
    Decoder: 32  → 64 → 256 → 790 (mirror of encoder)
    
    Train to reconstruct input gene expression.
    Encoder learns general gene expression patterns.
    """
    def __init__(self, input_dim=790, latent_dim=32, dropout=0.3):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, latent_dim)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )
    
    def forward(self, x):
        z = self.encoder(x)
        x_reconstructed = self.decoder(z)
        return x_reconstructed, z
    
    def encode(self, x):
        return self.encoder(x)


class PancancerDataset(Dataset):
    def __init__(self, data):
        self.data = torch.FloatTensor(data)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


# Scale pan-cancer data
scaler_pretrain = StandardScaler()
pancancer_scaled = scaler_pretrain.fit_transform(pancancer.values)

print(f"Pan-cancer scaled shape: {pancancer_scaled.shape}")
print(f"Mean: {pancancer_scaled.mean():.4f}")
print(f"Std:  {pancancer_scaled.std():.4f}")

# Test autoencoder
ae = ExpressionAutoencoder(input_dim=790).to(device)
test_input = torch.randn(4, 790).to(device)
recon, latent = ae(test_input)
print(f"\nAutoencoder test:")
print(f"  Input:       {test_input.shape}")
print(f"  Latent:      {latent.shape}")
print(f"  Reconstructed: {recon.shape}")
print("Autoencoder defined ✅")

Device: cpu
Pan-cancer scaled shape: (10535, 790)
Mean: 0.0000
Std:  1.0000

Autoencoder test:
  Input:       torch.Size([4, 790])
  Latent:      torch.Size([4, 32])
  Reconstructed: torch.Size([4, 790])
Autoencoder defined ✅


In [9]:
from sklearn.model_selection import train_test_split

# Split pan-cancer into train/val for pretraining
X_train, X_val = train_test_split(pancancer_scaled, test_size=0.1, random_state=42)

train_dataset = PancancerDataset(X_train)
val_dataset   = PancancerDataset(X_val)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False)

print(f"Train: {len(train_dataset)} patients")
print(f"Val:   {len(val_dataset)} patients")

# Pretrain
ae = ExpressionAutoencoder(input_dim=790).to(device)
optimizer = torch.optim.Adam(ae.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
criterion = nn.MSELoss()

best_val_loss  = np.inf
best_weights   = None
patience_counter = 0
patience       = 15
epochs         = 100

print(f"\nPretraining autoencoder on {len(train_dataset)} pan-cancer patients...")
print(f"{'Epoch':<8} {'Train Loss':<14} {'Val Loss':<14}")
print("-" * 36)

for epoch in range(1, epochs + 1):
    # Train
    ae.train()
    train_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, _ = ae(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    
    # Validate
    ae.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            recon, _ = ae(batch)
            val_loss += criterion(recon, batch).item()
    val_loss /= len(val_loader)
    
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_weights     = {k: v.clone() for k, v in ae.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:<8} {train_loss:<14.4f} {val_loss:<14.4f}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break

ae.load_state_dict(best_weights)
print(f"\nPretraining done! Best val loss: {best_val_loss:.4f}")

Train: 9481 patients
Val:   1054 patients

Pretraining autoencoder on 9481 pan-cancer patients...
Epoch    Train Loss     Val Loss      
------------------------------------
1        0.8386         0.6730        
5        0.5505         0.5056        
10       0.4993         0.4581        
15       0.4725         0.4351        
20       0.4606         0.4196        
25       0.4547         0.4095        
30       0.4459         0.4014        
35       0.4360         0.3960        
40       0.4356         0.3905        
45       0.4252         0.3878        
50       0.4232         0.3841        
55       0.4204         0.3804        
60       0.4218         0.3820        
65       0.4144         0.3787        
70       0.4136         0.3777        
75       0.4167         0.3750        
80       0.4077         0.3709        
85       0.4042         0.3686        
90       0.4025         0.3679        
95       0.3989         0.3669        
100      0.4011         0.3666        

Pretra

In [11]:
import pickle

base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'

# Save full autoencoder
torch.save(ae.state_dict(), 
           f'{base}/models/experiments/pretrained_autoencoder.pt')

# Save just the encoder weights separately
encoder_weights = {k.replace('encoder.', ''): v 
                   for k, v in ae.state_dict().items() 
                   if k.startswith('encoder.')}
torch.save(encoder_weights, 
           f'{base}/models/experiments/pretrained_encoder_weights.pt')

# Save the pretrain scaler
with open(f'{base}/models/experiments/scaler_pretrain.pkl', 'wb') as f:
    pickle.dump(scaler_pretrain, f)

# Save which genes were used
with open(f'{base}/models/experiments/pretrain_genes.json', 'w') as f:
    json.dump(list(pancancer.columns), f)

print("Saved:")
print(f"  models/experiments/pretrained_autoencoder.pt")
print(f"  models/experiments/pretrained_encoder_weights.pt")
print(f"  models/experiments/scaler_pretrain.pkl")
print(f"  models/experiments/pretrain_genes.json")
print(f"\nPretrained on: 9,481 pan-cancer patients")
print(f"Gene space:    {len(pancancer.columns)} genes")
print(f"Best val loss: {best_val_loss:.4f}")

Saved:
  models/experiments/pretrained_autoencoder.pt
  models/experiments/pretrained_encoder_weights.pt
  models/experiments/scaler_pretrain.pkl
  models/experiments/pretrain_genes.json

Pretrained on: 9,481 pan-cancer patients
Gene space:    790 genes
Best val loss: 0.3664


In [12]:
import sys
sys.path.append(f'{base}/notebooks')

# Load our data
expr     = pd.read_csv(f'{base}/data/processed/expression_matrix.csv', index_col=0)
dysreg   = pd.read_csv(f'{base}/data/processed/dysregulation_scores.csv', index_col=0)
immune   = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)
clinical = pd.read_csv(f'{base}/data/processed/clinical_survival.csv', index_col=0)

# Align patients
common   = expr.index.intersection(dysreg.index).intersection(
           immune.index).intersection(clinical.index)
expr     = expr.loc[common]
dysreg   = dysreg.loc[common]
immune   = immune.loc[common]
clinical = clinical.loc[common]

# Clinical features
age           = clinical[['age']].copy()
gender        = (clinical['gender'] == 'male').astype(float).to_frame()
stage_dummies = pd.get_dummies(clinical['stage_group'], prefix='stage')
stage_dummies = stage_dummies.drop(columns=['stage_Stage I'], errors='ignore')
clinical_features = pd.concat([age, gender, stage_dummies], axis=1).astype(float).fillna(0)

# Survival labels
y = np.array(
    [(bool(e), t) for e, t in zip(clinical['event'], clinical['survival_time'])],
    dtype=[('event', bool), ('time', float)]
)

# Filter expression to our 790 pretrained genes only
pretrain_genes = json.load(open(f'{base}/models/experiments/pretrain_genes.json'))
common_genes   = [g for g in pretrain_genes if g in expr.columns]
expr_790       = expr[common_genes]

print(f"Patients:           {len(common)}")
print(f"Expression genes:   {expr_790.shape[1]} (pretrained gene space)")
print(f"Dysregulation:      {dysreg.shape[1]}")
print(f"Immune:             {immune.shape[1]}")
print(f"Clinical:           {clinical_features.shape[1]}")
print(f"Events:             {y['event'].sum()} ({y['event'].mean()*100:.1f}%)")

Patients:           478
Expression genes:   790 (pretrained gene space)
Dysregulation:      819
Immune:             22
Clinical:           5
Events:             121 (25.3%)


In [13]:
class FusionModelPretrained(nn.Module):
    """
    FusionModelV3 with pretrained expression encoder.
    
    Key difference vs V3:
    - Expression encoder initialised with pretrained weights
    - Input dim = 790 (pretrained gene space) instead of 30
    - Everything else identical
    """
    def __init__(self, expr_dim=790, dysreg_dim=20,
                 immune_dim=22, clinical_dim=5, dropout=0.5):
        super().__init__()
        
        # Expression encoder — same architecture as autoencoder encoder
        self.encoder_expr = nn.Sequential(
            nn.Linear(expr_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32)
        )
        
        self.encoder_dysreg = nn.Sequential(
            nn.Linear(dysreg_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32)
        )
        self.encoder_immune = nn.Sequential(
            nn.Linear(immune_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )
        self.encoder_clinical = nn.Sequential(
            nn.Linear(clinical_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32)
        )
        self.attention = nn.Sequential(
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1)
        )
        self.output = nn.Linear(32, 1)
    
    def load_pretrained_encoder(self, weights_path):
        """Load pretrained encoder weights into expression encoder."""
        pretrained = torch.load(weights_path, map_location=device)
        self.encoder_expr.load_state_dict(pretrained)
        print("Pretrained encoder weights loaded ✅")
    
    def forward(self, x_expr, x_dysreg, x_immune, x_clinical):
        h_expr     = self.encoder_expr(x_expr)
        h_dysreg   = self.encoder_dysreg(x_dysreg)
        h_immune   = self.encoder_immune(x_immune)
        h_clinical = self.encoder_clinical(x_clinical)
        
        streams      = torch.stack([h_expr, h_dysreg, h_immune, h_clinical], dim=1)
        attn_weights = torch.softmax(self.attention(streams), dim=1)
        fused        = (attn_weights * streams).sum(dim=1)
        
        return self.output(fused), attn_weights.squeeze(-1)


# Test
model_test = FusionModelPretrained().to(device)
model_test.load_pretrained_encoder(
    f'{base}/models/experiments/pretrained_encoder_weights.pt')

r, a = model_test(
    torch.randn(4, 790).to(device),
    torch.randn(4, 20).to(device),
    torch.randn(4, 22).to(device),
    torch.randn(4, 5).to(device)
)
print(f"Model test passed — risk: {r.shape}, attn: {a.shape}")

Pretrained encoder weights loaded ✅
Model test passed — risk: torch.Size([4, 1]), attn: torch.Size([4, 4])


In [15]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SurvivalDataset(Dataset):
    def __init__(self, expr, dysreg, immune, clinical, times, events):
        self.expr     = torch.FloatTensor(expr)
        self.dysreg   = torch.FloatTensor(dysreg)
        self.immune   = torch.FloatTensor(immune)
        self.clinical = torch.FloatTensor(clinical)
        self.times    = torch.FloatTensor(times)
        self.events   = torch.FloatTensor(events)
    def __len__(self):
        return len(self.times)
    def __getitem__(self, idx):
        return (self.expr[idx], self.dysreg[idx],
                self.immune[idx], self.clinical[idx],
                self.times[idx], self.events[idx])


def cox_loss(risk_scores, times, events):
    order       = torch.argsort(times, descending=True)
    risk_scores = risk_scores[order].squeeze()
    events      = events[order]
    log_cumsum  = torch.logcumsumexp(risk_scores, dim=0)
    return -torch.mean((risk_scores - log_cumsum)[events.bool()])


def train_model(model, train_loader, val_expr, val_dysreg,
                val_immune, val_clinical, val_times, val_events,
                epochs=300, patience=30, lr=0.001, noise=0.05):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    best_val_cindex  = 0
    best_weights     = None
    patience_counter = 0
    for epoch in range(epochs):
        model.train()
        for x_expr, x_dysreg, x_immune, x_clinical, times, events in train_loader:
            x_expr     = x_expr.to(device)
            x_dysreg   = x_dysreg.to(device)
            x_immune   = x_immune.to(device)
            x_clinical = x_clinical.to(device)
            times      = times.to(device)
            events     = events.to(device)
            x_expr   = x_expr   + torch.randn_like(x_expr)   * noise
            x_dysreg = x_dysreg + torch.randn_like(x_dysreg) * noise
            x_immune = x_immune + torch.randn_like(x_immune) * noise
            optimizer.zero_grad()
            risk, _ = model(x_expr, x_dysreg, x_immune, x_clinical)
            loss = cox_loss(risk, times, events)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_risk, _ = model(
                val_expr.to(device), val_dysreg.to(device),
                val_immune.to(device), val_clinical.to(device))
            val_risk = val_risk.squeeze().cpu().numpy()
        val_ci = concordance_index_censored(
            val_events.astype(bool), val_times, val_risk)[0]
        scheduler.step(-val_ci)
        if val_ci > best_val_cindex:
            best_val_cindex  = val_ci
            best_weights     = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= patience:
            break
    model.load_state_dict(best_weights)
    return model, best_val_cindex, epoch

print("All classes and functions defined ✅")
print(f"Device: {device}")

All classes and functions defined ✅
Device: cpu


In [16]:
from sksurv.metrics import concordance_index_censored
from sklearn.model_selection import StratifiedKFold
from lifelines import CoxPHFitter

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_cindex       = []
fold_attn_weights = []

print("Running leakage-free 5-fold CV — Pretrained Fusion Model")
print("Expression encoder pretrained on 9,481 pan-cancer patients")
print(f"{'Fold':<6} {'Best Epoch':<12} {'Test C-index':<12}")
print("-" * 32)

for fold, (train_idx, test_idx) in enumerate(kf.split(expr_790, y['event']), 1):

    expr_train,     expr_test     = expr_790.iloc[train_idx],          expr_790.iloc[test_idx]
    dysreg_train,   dysreg_test   = dysreg.iloc[train_idx],            dysreg.iloc[test_idx]
    immune_train,   immune_test   = immune.iloc[train_idx],            immune.iloc[test_idx]
    clinical_train, clinical_test = clinical_features.iloc[train_idx], clinical_features.iloc[test_idx]
    y_train,        y_test        = y[train_idx],                      y[test_idx]

    times_train  = y_train['time'].copy()
    events_train = y_train['event'].copy()
    times_test   = y_test['time'].copy()
    events_test  = y_test['event'].copy()

    # Cox-guided dysregulation gene selection on training only
    cox_pvals_dysreg = {}
    for gene in dysreg_train.columns:
        try:
            df_tmp = pd.DataFrame({'T': times_train, 'E': events_train,
                                   'gene': dysreg_train[gene].values})
            cph = CoxPHFitter()
            cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
            cox_pvals_dysreg[gene] = cph.summary['p'].values[0]
        except:
            cox_pvals_dysreg[gene] = 1.0
    top_dysreg_genes = pd.Series(cox_pvals_dysreg).nsmallest(20).index

    dysreg_train_sel = dysreg_train[top_dysreg_genes]
    dysreg_test_sel  = dysreg_test[top_dysreg_genes]

    # Scale on training only
    # Use pretrain scaler for expression (same scale as pretraining)
    scaler_expr     = StandardScaler()
    scaler_dysreg   = StandardScaler()
    scaler_immune   = StandardScaler()
    scaler_clinical = StandardScaler()

    expr_train_s     = scaler_expr.fit_transform(expr_train)
    expr_test_s      = scaler_expr.transform(expr_test)
    dysreg_train_s   = scaler_dysreg.fit_transform(dysreg_train_sel)
    dysreg_test_s    = scaler_dysreg.transform(dysreg_test_sel)
    immune_train_s   = scaler_immune.fit_transform(immune_train)
    immune_test_s    = scaler_immune.transform(immune_test)
    clinical_train_s = scaler_clinical.fit_transform(clinical_train)
    clinical_test_s  = scaler_clinical.transform(clinical_test)

    val_size     = int(0.2 * len(train_idx))
    val_expr     = torch.FloatTensor(expr_train_s[:val_size])
    val_dysreg   = torch.FloatTensor(dysreg_train_s[:val_size])
    val_immune   = torch.FloatTensor(immune_train_s[:val_size])
    val_clinical = torch.FloatTensor(clinical_train_s[:val_size])
    val_times    = times_train[:val_size].copy()
    val_events   = events_train[:val_size].copy()

    train_dataset = SurvivalDataset(
        expr_train_s, dysreg_train_s, immune_train_s, clinical_train_s,
        times_train.copy(), events_train.copy())
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Load model with pretrained encoder
    model = FusionModelPretrained(
        expr_dim=790, dysreg_dim=20, immune_dim=22, clinical_dim=5
    ).to(device)
    model.load_pretrained_encoder(
        f'{base}/models/experiments/pretrained_encoder_weights.pt')

    model, best_val_ci, best_epoch = train_model(
        model, train_loader,
        val_expr, val_dysreg, val_immune, val_clinical,
        val_times, val_events,
        epochs=300, patience=30, lr=0.001, noise=0.05
    )

    model.eval()
    with torch.no_grad():
        test_risk, test_attn = model(
            torch.FloatTensor(expr_test_s).to(device),
            torch.FloatTensor(dysreg_test_s).to(device),
            torch.FloatTensor(immune_test_s).to(device),
            torch.FloatTensor(clinical_test_s).to(device)
        )

    test_risk = test_risk.squeeze().cpu().numpy()
    test_attn = test_attn.cpu().numpy()

    ci_test = concordance_index_censored(
        events_test.astype(bool), times_test, test_risk)[0]

    fold_cindex.append(ci_test)
    fold_attn_weights.append(test_attn)

    print(f"{fold:<6} {best_epoch:<12} {ci_test:.4f}")

print("-" * 32)
print(f"\nPretrained Fusion Model C-index: {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")
print(f"\nFull comparison:")
print(f"  Cox Clinical:                    0.700")
print(f"  Cox-Lasso Expression:            0.649")
print(f"  Fusion V3 (no pretrain):         0.655")
print(f"  Fusion V3 (augmentation):        0.669")
print(f"  Fusion (pretrained encoder):     {np.mean(fold_cindex):.3f}")

Running leakage-free 5-fold CV — Pretrained Fusion Model
Expression encoder pretrained on 9,481 pan-cancer patients
Fold   Best Epoch   Test C-index
--------------------------------


/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean 

Pretrained encoder weights loaded ✅
1      91           0.5531


/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean 

Pretrained encoder weights loaded ✅
2      85           0.7623


...

Pretrained encoder weights loaded ✅
3      80           0.5903


/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean 

Pretrained encoder weights loaded ✅
4      97           0.6099


/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['gene'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean 

Pretrained encoder weights loaded ✅
5      82           0.6930
--------------------------------

Pretrained Fusion Model C-index: 0.642 ± 0.076

Full comparison:
  Cox Clinical:                    0.700
  Cox-Lasso Expression:            0.649
  Fusion V3 (no pretrain):         0.655
  Fusion V3 (augmentation):        0.669
  Fusion (pretrained encoder):     0.642
